# Lab 04: Analyse trade data with pandas (solution)

This lab brings together Day 1: load several datasets, select, filter, sort, calculate, group, check quality and produce a summary report, one small step at a time.

---
# Part A: Load and inspect

## A1. Set up
Imports and folder paths.

*Run this cell; no changes needed.*

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

## A2. Load three datasets
Load `trade_summary.csv` into `trade`, `tariffs_mfn.csv` into `tariffs`, and the `countries` sheet of `countries.xlsx` into `countries` (as in Lab 03).

In [ ]:
trade = pd.read_csv(DATA / "trade_summary.csv")
tariffs = pd.read_csv(DATA / "tariffs_mfn.csv")
countries = pd.read_excel(DATA / "countries.xlsx", sheet_name="countries")

In [ ]:
# check
assert trade.shape == (280, 8) and tariffs.shape == (280, 4) and len(countries) == 14
print("A2 OK")

## A3. info()
`info()` shows every column, its type and how many values are present. Run it on every new dataset.

*Run this cell; no changes needed.*

In [ ]:
trade.info()

---
# Part B: Selecting columns

## B1. One column
`df["name"]` selects one column as a **Series**. Store the exports column in `exports`.

**Example**
```python
years = trade["year"]
```

In [ ]:
exports = trade["exports_usd_m"]
exports.head()

In [ ]:
# check
assert isinstance(exports, pd.Series) and len(exports) == 280
print("B1 OK")

## B2. Several columns
A **list** of names inside the brackets selects several columns: note the double brackets. Create `small` with `reporter`, `year` and `exports_usd_m`.

**Example**
```python
trade[["reporter", "region"]]
```

In [ ]:
small = trade[["reporter", "year", "exports_usd_m"]]
small.head()

In [ ]:
# check
assert list(small.columns) == ["reporter", "year", "exports_usd_m"]
print("B2 OK")

## B3. Statistics on a column
A Series has methods such as `.sum()`, `.mean()` and `.max()`. Calculate `total`, `average` and `largest` for `exports`.

In [ ]:
total = exports.sum()
average = exports.mean()
largest = exports.max()
print(f"{total:,.0f} {average:,.0f} {largest:,.0f}")

In [ ]:
# check
assert round(average * 280) == round(total)
print("B3 OK")

---
# Part C: Filtering rows

## C1. One condition
`trade["reporter"] == "Kenya"` gives True or False for every row. Put it inside `trade[...]` to keep only the True rows. Create `kenya`.

**Example**
```python
africa = trade[trade["region"] == "Africa"]
```

In [ ]:
kenya = trade[trade["reporter"] == "Kenya"]
kenya.head()

In [ ]:
# check
assert len(kenya) == 20
print("C1 OK")

## C2. Two conditions
Combine conditions with `&` (and). Wrap **each** condition in brackets. Create `kenya_2023`.

**Example**
```python
trade[(trade["region"] == "Africa") & (trade["year"] == 2020)]
```

In [ ]:
kenya_2023 = trade[(trade["reporter"] == "Kenya") & (trade["year"] == 2023)]
kenya_2023

In [ ]:
# check
assert len(kenya_2023) == 4
print("C2 OK")

## C3. Filter and choose columns
`trade.loc[rows, columns]` filters rows and picks columns in one go. Show Kenya's `year`, `product_group` and `exports_usd_m`.

In [ ]:
cols = ["year", "product_group", "exports_usd_m"]
kenya_view = trade.loc[trade["reporter"] == "Kenya", cols]
kenya_view.head()

In [ ]:
# check
assert list(kenya_view.columns) == cols and len(kenya_view) == 20
print("C3 OK")

## C4. A numeric condition
How many rows have imports above 1,000,000 (USD 1 trillion)? Filter, then use `len()`.

In [ ]:
big_imports = trade[trade["imports_usd_m"] > 1_000_000]
print(len(big_imports))

In [ ]:
# check
assert (big_imports["imports_usd_m"] > 1_000_000).all()
print("C4 OK")

---
# Part D: Sorting

## D1. Sort largest first
Filter to Manufactures (`product_code == "MAN"`) in 2023, then `.sort_values("exports_usd_m", ascending=False)`. Store in `manuf_2023`.

In [ ]:
manuf_2023 = (trade[(trade["product_code"] == "MAN") & (trade["year"] == 2023)]
              .sort_values("exports_usd_m", ascending=False))
manuf_2023[["reporter", "exports_usd_m"]]

In [ ]:
# check
assert len(manuf_2023) == 14 and manuf_2023.iloc[0]["reporter"] == "China"
print("D1 OK")

## D2. Top three
`.head(3)` keeps the first three rows. Store the top three in `top3`.

In [ ]:
top3 = manuf_2023.head(3)
top3[["reporter", "exports_usd_m"]]

In [ ]:
# check
assert len(top3) == 3
print("D2 OK")

---
# Part E: Calculated fields

## E1. Trade balance
Assigning to a new column name creates it. The calculation runs on every row at once. Add `balance_usd_m` = exports minus imports.

**Example**
```python
trade["total_trade"] = trade["exports_usd_m"] + trade["imports_usd_m"]
```

In [ ]:
trade["balance_usd_m"] = trade["exports_usd_m"] - trade["imports_usd_m"]
trade[["reporter", "exports_usd_m", "imports_usd_m", "balance_usd_m"]].head()

In [ ]:
# check
assert (trade["balance_usd_m"] == trade["exports_usd_m"] - trade["imports_usd_m"]).all()
print("E1 OK")

## E2. Billions
Add `exports_usd_bn`: exports divided by 1000, rounded to 1 decimal place with `.round(1)`.

In [ ]:
trade["exports_usd_bn"] = (trade["exports_usd_m"] / 1000).round(1)
trade[["exports_usd_m", "exports_usd_bn"]].head()

In [ ]:
# check
assert trade.loc[0, "exports_usd_bn"] == round(trade.loc[0, "exports_usd_m"] / 1000, 1)
print("E2 OK")

## E3. A True/False column
Add `surplus`: True where `balance_usd_m` is greater than 0. Then count the surpluses with `.sum()` (True counts as 1).

In [ ]:
trade["surplus"] = trade["balance_usd_m"] > 0
print(trade["surplus"].sum())

In [ ]:
# check
assert trade["surplus"].dtype == bool
print("E3 OK")

---
# Part F: Grouping and summarising

## F1. Total by year
`groupby("year")` splits the rows by year; then pick a column and `.sum()`. Store in `by_year`.

**Example**
```python
trade.groupby("region")["imports_usd_m"].sum()
```

In [ ]:
by_year = trade.groupby("year")["exports_usd_m"].sum()
by_year

In [ ]:
# check
assert len(by_year) == 5 and by_year[2020] < by_year[2019]
print("F1 OK")

## F2. Top five exporters in 2023
Filter to 2023, group by `reporter`, sum exports, sort largest first and keep five. Store in `top5_2023`.

In [ ]:
top5_2023 = (trade[trade["year"] == 2023]
             .groupby("reporter")["exports_usd_m"].sum()
             .sort_values(ascending=False)
             .head(5))
top5_2023

In [ ]:
# check
assert len(top5_2023) == 5 and top5_2023.index[0] == "China"
print("F2 OK")

## F3. A pivot table
`pivot_table` builds a cross-tab like Excel. Create `region_year` with regions as rows, years as columns and summed exports.

**Example**
```python
trade.pivot_table(index="reporter", columns="year",
                  values="imports_usd_m", aggfunc="sum")
```

In [ ]:
region_year = trade.pivot_table(index="region", columns="year",
                                values="exports_usd_m", aggfunc="sum")
region_year.round(0)

In [ ]:
# check
assert region_year.shape == (5, 5)
print("F3 OK")

## F4. Growth from 2019 to 2023
Make a reporter-by-year pivot called `totals` (like F3, but `index="reporter"`). Then `growth = (totals[2023] / totals[2019] - 1) * 100`, rounded to 1 dp and sorted largest first.

In [ ]:
totals = trade.pivot_table(index="reporter", columns="year",
                           values="exports_usd_m", aggfunc="sum")
growth = ((totals[2023] / totals[2019] - 1) * 100).round(1).sort_values(ascending=False)
growth

In [ ]:
# check
assert len(growth) == 14
print("F4 OK")

## F5. Agricultural share in 2023
1. `t23` = the 2023 rows.
2. `agr` = agricultural exports (`product_code == "AGR"`) per reporter: filter, group, sum.
3. `total` = all exports per reporter in `t23`.
4. `agr_share = (agr / total * 100).round(1)`, sorted largest first.

In [ ]:
t23 = trade[trade["year"] == 2023]
agr = t23[t23["product_code"] == "AGR"].groupby("reporter")["exports_usd_m"].sum()
total = t23.groupby("reporter")["exports_usd_m"].sum()
agr_share = (agr / total * 100).round(1).sort_values(ascending=False)
agr_share

In [ ]:
# check
assert len(agr_share) == 14 and agr_share.between(0, 100).all()
print("F5 OK")

---
# Part G: Data quality

## G1. Check each dataset
Missing values, duplicate rows and the smallest value in each numeric column.

*Run this cell; no changes needed.*

In [ ]:
for name, df in [("trade", trade), ("tariffs", tariffs), ("countries", countries)]:
    print("==", name)
    print("missing values:", df.isna().sum().sum())
    print("duplicate rows:", df.duplicated().sum())
    print(df.select_dtypes("number").min(), end="\n\n")

**Question:** What quality issues did you find? Why might they matter for an average tariff?

*Your answer:* `trade` and `countries` are complete with no duplicates or negative values. `tariffs` has some missing values; averages will silently skip them, so an average could cover fewer years for some countries.

---
# Part H: Combine and report

## H1. Add income group
`merge` joins two tables on a key (Module 07 covers this in depth). Add `income_group` from `countries` to `trade`.

**Example**
```python
trade.merge(countries[["iso3", "region"]], left_on="reporter_iso3", right_on="iso3", how="left")
```

In [ ]:
merged = trade.merge(countries[["iso3", "income_group"]],
                     left_on="reporter_iso3", right_on="iso3", how="left")
merged[["reporter", "income_group"]].head()

In [ ]:
# check
assert merged["income_group"].notna().all() and len(merged) == 280
print("H1 OK")

## H2. Exports by income group, 2023
Filter `merged` to 2023, group by `income_group` and sum exports. Store in `by_income`.

In [ ]:
by_income = merged[merged["year"] == 2023].groupby("income_group")["exports_usd_m"].sum()
by_income

In [ ]:
# check
assert len(by_income) == 3
print("H2 OK")

## H3. Write the summary workbook
`pd.ExcelWriter` writes several sheets into one workbook. Add sheets for `top5_2023`, `growth` and `agr_share`, following the first line.

In [ ]:
with pd.ExcelWriter(OUT / "day1_summary.xlsx") as writer:
    region_year.to_excel(writer, sheet_name="region_year")
    top5_2023.to_excel(writer, sheet_name="top5_2023")
    growth.to_excel(writer, sheet_name="growth")
    agr_share.to_excel(writer, sheet_name="agr_share")

In [ ]:
# check
sheets = pd.ExcelFile(OUT / "day1_summary.xlsx").sheet_names
assert {"region_year", "top5_2023", "growth", "agr_share"} <= set(sheets)
print("H3 OK")

**Question:** Write three bullet-point findings for a manager, based on your summaries.

*Your answer:* - Exports in every region dipped in 2020 and exceeded 2019 levels by 2021-2022.
- China is by far the largest exporter, around double the next economy.
- Agricultural products are a noticeably larger share of exports for some lower middle income economies. (Illustrative data.)